<a href="https://colab.research.google.com/github/aayush-jain-dtu/inventory-stock-prediction/blob/main/covid_comparison_model_adaptive_vs_Frozen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
aml_backtesting_frozen_vs_adaptive.py
======================================
Empirical Proof of Adaptivity — Stockify Major Project

Two models tested on identical monthly windows (Jan 2020 → Dec 2022):

  FROZEN MODEL   : Trained ONCE on 2014–2019. Never retrained. Represents
                   the "static model" failure mode from the literature.

  ADAPTIVE MODEL : Initially trained on 2014–2019 (same start). After each
                   month's test, the training window EXPANDS to include that
                   month, and the model is retrained before the next test.
                   This is the Expanding Window (Walk-Forward) strategy.

OUTPUT
------
  • adaptive_vs_frozen_results.csv   — monthly metrics for both models
  • adaptive_vs_frozen_plot.png      — dual-panel figure

"""


'\naml_backtesting_frozen_vs_adaptive.py\n======================================\nEmpirical Proof of Adaptivity — Stockify Major Project\n \nTwo models tested on identical monthly windows (Jan 2020 → Dec 2022):\n \n  FROZEN MODEL   : Trained ONCE on 2014–2019. Never retrained. Represents\n                   the "static model" failure mode from the literature.\n \n  ADAPTIVE MODEL : Initially trained on 2014–2019 (same start). After each\n                   month\'s test, the training window EXPANDS to include that\n                   month, and the model is retrained before the next test.\n                   This is the Expanding Window (Walk-Forward) strategy.\n \nOUTPUT\n------\n  • adaptive_vs_frozen_results.csv   — monthly metrics for both models\n  • adaptive_vs_frozen_plot.png      — dual-panel figure\n \n'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# 1.  LOAD DATA

In [3]:
CSV_PATH = "inventory_dataset_covid_drift.csv"

print("=" * 62)
print("  STOCKIFY — AML BACKTESTING: Frozen vs Adaptive HGB")
print("=" * 62)

df_raw = pd.read_csv(CSV_PATH)
print(f"\n  Loaded {len(df_raw):,} rows  |  {df_raw['order_year'].min()}–{df_raw['order_year'].max()}")

  STOCKIFY — AML BACKTESTING: Frozen vs Adaptive HGB

  Loaded 8,000 rows  |  2014–2022


# 2.  PREPROCESSING & FEATURE ENGINEERING

In [4]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies the full preprocessing pipeline described in Chapter 3:
      - Column removal (order_id, client_id, product_title, demand_phase)
      - days_from_start  (long-term trend proxy)
      - week_of_month    (intra-month cyclical feature)
      - OHE on product_id, product_category, order_month, week_of_month
    Returns a clean feature-matrix DataFrame (no scaling yet — scaling is
    done per split to avoid data leakage).
    """
    df = df.copy()

    # Reconstruct a single date column for arithmetic
    df["order_date"] = pd.to_datetime(
        dict(year=df["order_year"], month=df["order_month"], day=df["order_day"])
    )

    origin = pd.Timestamp("2014-01-01")
    df["days_from_start"] = (df["order_date"] - origin).dt.days

    # week_of_month (1–5 buckets)
    df["week_of_month"] = ((df["order_day"] - 1) // 7 + 1).clip(1, 5)

    # Drop columns with no predictive value or redundant info
    drop_cols = ["order_id", "client_id", "product_title",
                 "demand_phase", "order_date"]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    # One-Hot Encoding
    ohe_cols = ["product_id", "product_category", "order_month", "week_of_month"]
    df = pd.get_dummies(df, columns=ohe_cols, drop_first=False)

    return df


df = build_features(df_raw)

TARGET   = "quantity_ordered"
FEATURES = [c for c in df.columns if c != TARGET]

# Sort by time so expanding window respects chronological order
df_raw["order_date"] = pd.to_datetime(
    dict(year=df_raw["order_year"], month=df_raw["order_month"], day=df_raw["order_day"])
)
df["order_date"] = df_raw["order_date"].values
df.sort_values("order_date", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"  Feature columns after engineering: {len(FEATURES)}")

  Feature columns after engineering: 43


# 3.  DEFINE SPLITS
#
#     Initial training  : 2014-01-01 → 2019-12-31  (both models)
#     Test window       : Jan 2020 → Dec 2022  (month-by-month)

In [6]:
TRAIN_END    = pd.Timestamp("2019-12-31")
TEST_START   = pd.Timestamp("2020-01-01")
TEST_END     = pd.Timestamp("2022-12-31")

mask_init_train = df["order_date"] <= TRAIN_END
mask_test_range = (df["order_date"] >= TEST_START) & (df["order_date"] <= TEST_END)

X_all = df[FEATURES]
y_all = df[TARGET]

# Collect all (year, month) pairs in the test range
test_months = pd.PeriodIndex(
    df.loc[mask_test_range, "order_date"]
    .dt.to_period("M")
    .unique()
).sort_values()
print(f"\n  Initial training rows : {mask_init_train.sum():,}")
print(f"  Test months           : {len(test_months)}  ({test_months[0]} → {test_months[-1]})")



  Initial training rows : 5,380
  Test months           : 36  (2020-01 → 2022-12)


# 4.  HGB HYPERPARAMETERS
#     Same config for both models — fair comparison

In [7]:
HGB_PARAMS = dict(
    max_iter       = 300,
    learning_rate  = 0.05,
    max_depth      = 6,
    min_samples_leaf = 20,
    l2_regularization = 0.1,
    random_state   = 42,
)

# 5.  TRAIN FROZEN MODEL (one-time fit on 2014–2019)

In [8]:
print("\n  [1/3] Training FROZEN model on 2014–2019 …")

X_init = X_all.loc[mask_init_train]
y_init = y_all.loc[mask_init_train]

# Scale using only initial training data
scaler_frozen = StandardScaler()
X_init_scaled = scaler_frozen.fit_transform(X_init)
y_init_arr    = y_init.values
y_std_frozen  = y_init_arr.std()          # σ of training labels for RMSE/σ

frozen_model = HistGradientBoostingRegressor(**HGB_PARAMS)
frozen_model.fit(X_init_scaled, y_init_arr)
print("    Frozen model trained ✓")


  [1/3] Training FROZEN model on 2014–2019 …
    Frozen model trained ✓


# 6.  EXPANDING-WINDOW BACKTESTING LOOP

In [9]:
print("\n  [2/3] Running expanding-window backtesting …\n")
print(f"  {'Month':<12} {'Frozen R²':>10} {'Frozen RMSE/σ':>14} "
      f"{'Adapt R²':>10} {'Adapt RMSE/σ':>13} {'Δ RMSE/σ':>10}")
print("  " + "─" * 75)

results = []

# Adaptive model starts with a copy of the frozen model's scaler&fit
# but will retrain progressively.  We keep a running "seen" mask.
adaptive_seen_mask = mask_init_train.copy()

# Initial adaptive scaler/model = same as frozen
scaler_adaptive = StandardScaler()
scaler_adaptive.fit(X_init)
y_std_adaptive  = y_init_arr.std()

adaptive_model = HistGradientBoostingRegressor(**HGB_PARAMS)
adaptive_model.fit(scaler_adaptive.transform(X_init), y_init_arr)

for period in test_months:
    # ── Locate this month's rows ──────────────────────────────────────────
    month_mask = df["order_date"].dt.to_period("M") == period
    if month_mask.sum() < 5:          # skip sparse months
        continue

    X_month = X_all.loc[month_mask]
    y_month = y_all.loc[month_mask].values

    # ── FROZEN: predict with stale scaler + frozen model ─────────────────
    X_month_frozen = scaler_frozen.transform(X_month)
    y_pred_frozen  = frozen_model.predict(X_month_frozen)

    rmse_frozen = mean_squared_error(y_month, y_pred_frozen) ** 0.5
    r2_frozen   = r2_score(y_month, y_pred_frozen)
    nrmse_frozen = rmse_frozen / y_std_frozen

    # ── ADAPTIVE: predict BEFORE retraining (test first, then learn) ──────
    X_month_adaptive = scaler_adaptive.transform(X_month)
    y_pred_adaptive  = adaptive_model.predict(X_month_adaptive)

    rmse_adaptive = mean_squared_error(y_month, y_pred_adaptive) ** 0.5
    r2_adaptive   = r2_score(y_month, y_pred_adaptive)
    nrmse_adaptive = rmse_adaptive / y_std_adaptive

    delta = nrmse_frozen - nrmse_adaptive          # positive = adaptive better

    print(f"  {str(period):<12} {r2_frozen:>10.3f} {nrmse_frozen:>14.3f} "
          f"{r2_adaptive:>10.3f} {nrmse_adaptive:>13.3f} {delta:>+10.3f}")

    results.append({
        "period"          : str(period),
        "year"            : period.year,
        "month"           : period.month,
        "frozen_r2"       : round(r2_frozen,    4),
        "frozen_nrmse"    : round(nrmse_frozen,  4),
        "adaptive_r2"     : round(r2_adaptive,   4),
        "adaptive_nrmse"  : round(nrmse_adaptive, 4),
        "delta_nrmse"     : round(delta,          4),
        "n_test_rows"     : int(month_mask.sum()),
    })

    # ── ADAPTIVE: expand window and retrain ───────────────────────────────
    adaptive_seen_mask |= month_mask      # add this month to training history

    X_train_exp = X_all.loc[adaptive_seen_mask]
    y_train_exp = y_all.loc[adaptive_seen_mask].values

    # Refit scaler on the now-larger training set (prevents leakage)
    scaler_adaptive = StandardScaler()
    scaler_adaptive.fit(X_train_exp)
    y_std_adaptive = y_train_exp.std()

    adaptive_model = HistGradientBoostingRegressor(**HGB_PARAMS)
    adaptive_model.fit(scaler_adaptive.transform(X_train_exp), y_train_exp)


  [2/3] Running expanding-window backtesting …

  Month         Frozen R²  Frozen RMSE/σ   Adapt R²  Adapt RMSE/σ   Δ RMSE/σ
  ───────────────────────────────────────────────────────────────────────────
  2020-01           0.608          0.650      0.608         0.650     +0.000
  2020-02           0.691          0.578      0.693         0.576     +0.002
  2020-03          -2.990          6.203     -3.023         6.218     -0.016
  2020-04          -2.898          6.047      0.302         2.157     +3.890
  2020-05          -2.686          5.982      0.366         1.851     +4.130
  2020-06          -3.164          5.902      0.515         1.353     +4.549
  2020-07          -2.352          5.816      0.459         1.448     +4.367
  2020-08          -3.216          6.225      0.478         1.281     +4.944
  2020-09          -2.677          6.189      0.390         1.398     +4.791
  2020-10          -1.826          5.624      0.344         1.433     +4.190
  2020-11          -3.480 

# 7.  SAVE RESULTS CSV

In [10]:
results_df = pd.DataFrame(results)
csv_out = "adaptive_vs_frozen_results.csv"
results_df.to_csv(csv_out, index=False)
print(f"\n  Results saved → {csv_out}")


  Results saved → adaptive_vs_frozen_results.csv


# 8.  SUMMARY STATISTICS

In [12]:
covid_mask    = (results_df["year"] == 2020) | \
                ((results_df["year"] == 2021) & (results_df["month"] <= 6))
recovery_mask = ~covid_mask

print("\n" + "=" * 62)
print("  SUMMARY")
print("=" * 62)
for label, mask in [("COVID shock  (Jan 2020 – Jun 2021)", covid_mask),
                    ("Recovery     (Jul 2021 – Dec 2022)", recovery_mask),
                    ("Full period  (Jan 2020 – Dec 2022)", pd.Series([True]*len(results_df)))]:
    sub = results_df[mask]
    print(f"\n  {label}")
    print(f"    Frozen   avg R² = {sub['frozen_r2'].mean():.3f}  |  avg RMSE/σ = {sub['frozen_nrmse'].mean():.3f}")
    print(f"    Adaptive avg R² = {sub['adaptive_r2'].mean():.3f}  |  avg RMSE/σ = {sub['adaptive_nrmse'].mean():.3f}")
    print(f"    Mean Δ RMSE/σ (frozen−adaptive) = {sub['delta_nrmse'].mean():+.3f}   "
          f"← {'Adaptive BETTER' if sub['delta_nrmse'].mean() > 0 else 'Frozen BETTER'}")



  SUMMARY

  COVID shock  (Jan 2020 – Jun 2021)
    Frozen   avg R² = -2.546  |  avg RMSE/σ = 5.435
    Adaptive avg R² = 0.258  |  avg RMSE/σ = 1.474
    Mean Δ RMSE/σ (frozen−adaptive) = +3.962   ← Adaptive BETTER

  Recovery     (Jul 2021 – Dec 2022)
    Frozen   avg R² = -0.647  |  avg RMSE/σ = 1.742
    Adaptive avg R² = -0.509  |  avg RMSE/σ = 0.591
    Mean Δ RMSE/σ (frozen−adaptive) = +1.151   ← Adaptive BETTER

  Full period  (Jan 2020 – Dec 2022)
    Frozen   avg R² = -1.596  |  avg RMSE/σ = 3.589
    Adaptive avg R² = -0.125  |  avg RMSE/σ = 1.032
    Mean Δ RMSE/σ (frozen−adaptive) = +2.557   ← Adaptive BETTER


# 9.  PUBLICATION-READY PLOT

In [13]:
print("\n  [3/3] Generating plot …")

# ── Colour palette ────────────────────────────────────────────────────────────
COL_FROZEN   = "#E24B4A"        # red  — frozen model
COL_ADAPTIVE = "#1D9E75"        # teal — adaptive model
COL_COVID    = "#E24B4A"        # phase band colour
COL_RECOVERY = "#378ADD"

periods      = results_df["period"].tolist()
x            = np.arange(len(periods))
tick_labels  = [p if p.endswith("-01") or p.endswith("-07") else "" for p in periods]

# ── Identify phase boundaries ─────────────────────────────────────────────────
covid_start_idx    = next(i for i, p in enumerate(periods) if "2020-01" in p)
recovery_start_idx = next(i for i, p in enumerate(periods) if "2021-07" in p)

fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)
fig.patch.set_facecolor("#fafafa")

def shade_phases(ax):
    """Shade background bands for COVID and Recovery phases."""
    ax.axvspan(covid_start_idx - 0.5,    recovery_start_idx - 0.5,
               alpha=0.07, color=COL_COVID, zorder=0)
    ax.axvspan(recovery_start_idx - 0.5, len(periods) - 0.5,
               alpha=0.07, color=COL_RECOVERY, zorder=0)
    # Phase boundary lines
    ax.axvline(covid_start_idx - 0.5,    color=COL_COVID,    lw=1.0, ls="--", alpha=0.5)
    ax.axvline(recovery_start_idx - 0.5, color=COL_RECOVERY, lw=1.0, ls="--", alpha=0.5)

# ── PANEL 1: RMSE/σ over time ─────────────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor("#ffffff")
shade_phases(ax1)
ax1.plot(x, results_df["frozen_nrmse"],   color=COL_FROZEN,   lw=2.2,
         marker="o", markersize=4.5, label="Frozen model",   zorder=3)
ax1.plot(x, results_df["adaptive_nrmse"], color=COL_ADAPTIVE, lw=2.2,
         marker="s", markersize=4.5, label="Adaptive model", zorder=3, ls="--")
ax1.fill_between(x,
                 results_df["adaptive_nrmse"],
                 results_df["frozen_nrmse"],
                 where=results_df["frozen_nrmse"] >= results_df["adaptive_nrmse"],
                 alpha=0.15, color=COL_ADAPTIVE, label="Adaptive advantage")
ax1.fill_between(x,
                 results_df["adaptive_nrmse"],
                 results_df["frozen_nrmse"],
                 where=results_df["frozen_nrmse"] < results_df["adaptive_nrmse"],
                 alpha=0.12, color=COL_FROZEN, label="Frozen advantage")
ax1.set_ylabel("RMSE / σ  (lower = better)", fontsize=11)
ax1.set_title("Normalised Prediction Error — Adaptive vs Frozen HGB",
              fontsize=13, fontweight="semibold", pad=8)
ax1.legend(loc="upper left", fontsize=9, framealpha=0.85)
ax1.grid(axis="y", alpha=0.3, lw=0.6)
ax1.spines[["top", "right"]].set_visible(False)

# Annotate phase labels
ax1.text(covid_start_idx + 0.5, ax1.get_ylim()[0] + 0.02,
         "COVID shock", color=COL_COVID, fontsize=8.5, alpha=0.7)
ax1.text(recovery_start_idx + 0.5, ax1.get_ylim()[0] + 0.02,
         "Recovery", color=COL_RECOVERY, fontsize=8.5, alpha=0.7)

# ── PANEL 2: R² over time ─────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor("#ffffff")
shade_phases(ax2)
ax2.plot(x, results_df["frozen_r2"],   color=COL_FROZEN,   lw=2.2,
         marker="o", markersize=4.5, label="Frozen R²")
ax2.plot(x, results_df["adaptive_r2"], color=COL_ADAPTIVE, lw=2.2,
         marker="s", markersize=4.5, label="Adaptive R²", ls="--")
ax2.axhline(0.70, color="#aaaaaa", lw=1.0, ls=":", label="Target R² = 0.70")
ax2.set_ylabel("R²  (higher = better)", fontsize=11)
ax2.set_title("Goodness-of-Fit (R²) — Adaptive vs Frozen HGB",
              fontsize=13, fontweight="semibold", pad=8)
ax2.legend(loc="lower left", fontsize=9, framealpha=0.85)
ax2.grid(axis="y", alpha=0.3, lw=0.6)
ax2.spines[["top", "right"]].set_visible(False)

# ── PANEL 3: Δ RMSE/σ (bar chart) ────────────────────────────────────────────
ax3 = axes[2]
ax3.set_facecolor("#ffffff")
shade_phases(ax3)
bar_colors = [COL_ADAPTIVE if v >= 0 else COL_FROZEN for v in results_df["delta_nrmse"]]
ax3.bar(x, results_df["delta_nrmse"], color=bar_colors, alpha=0.75, zorder=3)
ax3.axhline(0, color="#555555", lw=0.8)

# Rolling mean overlay
rolling_delta = results_df["delta_nrmse"].rolling(3, min_periods=1).mean()
ax3.plot(x, rolling_delta, color="#333333", lw=1.8, ls="--",
         label="3-month rolling mean", zorder=4)

ax3.set_ylabel("Δ RMSE/σ  (frozen − adaptive)", fontsize=11)
ax3.set_title("Monthly Adaptivity Gain  (positive = adaptive wins)",
              fontsize=13, fontweight="semibold", pad=8)
ax3.legend(loc="upper right", fontsize=9, framealpha=0.85)
ax3.grid(axis="y", alpha=0.3, lw=0.6)
ax3.spines[["top", "right"]].set_visible(False)

# Patch legend for bar colours
patch_a = mpatches.Patch(color=COL_ADAPTIVE, alpha=0.75, label="Adaptive better")
patch_f = mpatches.Patch(color=COL_FROZEN,   alpha=0.75, label="Frozen better")
ax3.legend(handles=[patch_a, patch_f,
                    plt.Line2D([0],[0], color="#333333", lw=1.8, ls="--",
                               label="3-month rolling mean")],
           loc="upper left", fontsize=9, framealpha=0.85)

# ── Shared x-axis labels ──────────────────────────────────────────────────────
ax3.set_xticks(x)
ax3.set_xticklabels(periods, rotation=55, ha="right", fontsize=7.5)
ax3.set_xlim(-0.5, len(periods) - 0.5)

plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.suptitle(
    "Empirical Proof of Adaptivity — Stockify Expanding-Window Backtesting\n"
    "Frozen HGB (trained 2014–2019, never retrained)  vs  "
    "Adaptive HGB (retrained each month, expanding window)",
    fontsize=12, y=0.995, color="#222222"
)

plot_out = "adaptive_vs_frozen_plot.png"
plt.savefig(plot_out, dpi=180, bbox_inches="tight", facecolor="#fafafa")
plt.close()

print(f"  Plot saved → {plot_out}")
print("\n" + "=" * 62)
print("  Done.  Use adaptive_vs_frozen_results.csv for your report table")
print("  and adaptive_vs_frozen_plot.png as Figure 5.x.")
print("=" * 62)


  [3/3] Generating plot …
  Plot saved → adaptive_vs_frozen_plot.png

  Done.  Use adaptive_vs_frozen_results.csv for your report table
  and adaptive_vs_frozen_plot.png as Figure 5.x.
